## Lecture du fichier de données
- 2500 lignes -> pandas suffisant
- Eviter les conversions silencieuses
- gestion de l'encodage du champ synthèse_entretien

In [61]:
import pandas as pd

In [62]:
data_file = "./dataset_trajectoire_emploi_Sujet Examen CISIA - Promo Upskilling Atlas - mai-oct2026 (Session-00279143).csv"
df = pd.read_csv(data_file, dtype={"age": "Int64", "code_postal": "category"})

## Pré analyse
- Types de données
- Recherche des manquants

In [63]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2500 entries, 0 to 2499
Data columns (total 10 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   usager_id             2500 non-null   str    
 1   age                   2378 non-null   Int64  
 2   niveau_diplome        2416 non-null   str    
 3   anciennete_poste_ans  2500 non-null   float64
 4   code_rome_vise        2500 non-null   str    
 5   code_insee_commune    2500 non-null   str    
 6   est_allocataire       2456 non-null   float64
 7   nationalite_hors_ue   2500 non-null   int64  
 8   synthese_entretien    2419 non-null   str    
 9   classe_retour_emploi  2500 non-null   int64  
dtypes: Int64(1), float64(2), int64(2), str(5)
memory usage: 197.9 KB


In [64]:
missing = pd.DataFrame({"nb_manquants": df.isna().sum(), "pct_manquants": (df.isna().mean() * 100).round(2)})

print(missing.sort_values("nb_manquants", ascending=False).head(10))

                      nb_manquants  pct_manquants
age                            122           4.88
niveau_diplome                  84           3.36
synthese_entretien              81           3.24
est_allocataire                 44           1.76
usager_id                        0           0.00
anciennete_poste_ans             0           0.00
code_insee_commune               0           0.00
code_rome_vise                   0           0.00
nationalite_hors_ue              0           0.00
classe_retour_emploi             0           0.00


In [ ]:
# Nombre de valeurs uniques des variables catégorielles ordinales
print(f"""Nombre catégories de la variable code_rome_vise : {df["code_rome_vise"].nunique()}""")
print(f"""Nombre catégories de la variable code_insee_commune : {df["code_insee_commune"].nunique()}""")

Nombre catégories de la variable code_rome_vise : 50
Nombre catégories de la variable code_rome_vise : 2407


- Un examen des données montre qu'il n'y a pas de manquants implicites 

## Conclusion préliminaire

| Nom feature | Type | Sensible | PII | Nature de la variable | Manquants (%) | Utilisation de la variable (premier avis) |
|------------|------|-----------|-----|----------------------------------|---------------|-----------|
| usager_id | str | non | identifiant pseudonymisé | catégorielle nominale | non | non |
| age | Int64 | oui | par croisement | numérique | 4.88 | à étudier |
| niveau_diplome | str | non | non | catégorielle ordinale | 3.36 | oui, manquants à gérer |
| anciennete_poste_ans | float64 | non | non | numérique | 0 | oui |
| code_rome_vise | str | non | non | ctégorielle nominale (50 valeurs uniques) | 0 | oui |
| code_insee_commune | str | oui | par croisement | catégorielle nominale (2407 valeurs uniques) | 0 | à étudier : variable très dispersée, one hot-encoding impossible, PII indirecte possible |
| est_allocataire | float64 | oui (métier) | non | numérique binaire | 1.76 | oui |
| nationalite_hors_ue | int64 | oui | par croisement | numérique binaire | 0 | à étudier, risque de discrimination |
| synthese_entretien | str | non | potentiellement oui | texte libre | 3.24 | à étudier : anonymisation requise? A transformer + gestion des manquants |
| Classe_retour_emploi | int64 | non | non | numérique | 0 | variable cible |



## Analyse de répartition des variables

In [66]:
num_cols = df.select_dtypes(include=["number"]).columns
cat_cols = df.select_dtypes(include=["string"]).columns
cat_cols = [col for col in cat_cols if col != "synthese_entretien"]

display(df[num_cols].describe().T)

,count,mean,std,min,25%,50%,75%,max
age,2378.0,40.566442,13.225242,18.0,29.0,41.0,52.0,63.0
anciennete_poste_ans,2500.0,2.9716,2.979548,0.0,0.8,2.0,4.1,22.6
est_allocataire,2456.0,0.595684,0.490859,0.0,0.0,1.0,1.0,1.0
nationalite_hors_ue,2500.0,0.118,0.322673,0.0,0.0,0.0,0.0,1.0
classe_retour_emploi,2500.0,0.8064,0.719671,0.0,0.0,1.0,1.0,2.0


In [67]:
# Analyse de la variable cible
display(
    df["classe_retour_emploi"]
    .value_counts(normalize=True)
    .sort_index()
    .mul(100)
    .round(2)
)

classe_retour_emploi
0    37.44
1    44.48
2    18.08
Name: proportion, dtype: float64

In [68]:
from scipy.stats import entropy
import numpy as np

rows = []
for col in cat_cols:
    p = df[col].value_counts(normalize=True)
    nb_modalites = len(p)
    pct_modalite_majoritaire = p.iloc[0] * 100
    pct_moyenne_theorique = 100 / nb_modalites
    ratio_concentration = (pct_modalite_majoritaire / pct_moyenne_theorique)

    entropie_normalisee = entropy(p) / np.log(nb_modalites)

    rows.append({"Variable": col,
                 "Nb modalités": nb_modalites,
                 "Catégorie majoritaire": p.index[0],
                 "% catégorie majoritaire": round(pct_modalite_majoritaire, 2),
                 "% moyen théorique": round(pct_moyenne_theorique, 2),
                 "Ratio concentration": round(ratio_concentration, 2),
                 "Entropie normalisée": round(entropie_normalisee, 3)})

cat_summary = pd.DataFrame(rows)

display(cat_summary.sort_values("Ratio concentration", ascending=False))

,Variable,Nb modalités,Catégorie majoritaire,% catégorie majoritaire,% moyen théorique,Ratio concentration,Entropie normalisée
3,code_insee_commune,2407,46283,0.12,0.04,2.89,0.998
1,niveau_diplome,4,Bac,39.98,25.00,1.60,0.961
2,code_rome_vise,50,H1206,2.84,2.00,1.42,0.997
0,usager_id,2500,ID_0000,0.04,0.04,1.00,1.000


## Conclusion

Variables catégorielle :
- usager_id : 1 numéro par ligne, identifiant OK (ne sera pas utilisé)
- code_insee_commune : 2407 valeurs différentes, peu de répétitions, bonne répartition des cas sur le territoire -> one hot encoding peu adapté
- code_rome_vise : bonne répartition des échantillons entre les 50 catégories
- niveau_diplome : le niveau Bac est majoritaire sans être écrasant

Variables numériques :
- Age : la moyenne étant proche de la médiane, la distribution est probablement symétrique autour de l'âge de 40 ans, avec un échantillon entre 18 et 63 ans qui semble représentatif de la population active
- anciennete_poste_ans : l'ancienneté moyenne est de 3 ans qui est supérieure à la valeur médiane qui est de 2 ans -> la distribution semble être asymétrique, la valeur maximale de 22.6 ans demande de vérifier la présence de valeurs aberrantes dans le dataset
- est_allocataire : variable binaire représentée à 60 / 40. Pas de déséquilibre excessif.
- nationalite_hors_ue : seuls 11.8% des personnes du dataset ont une origine hors UE. ANalyse plus poussée à réaliser pour cette variable pour éviter un biais et un effet discriminant pour le modèle

Variable cible :
- La répartition entre les 3 catégories est relativement équilibrée, le ratio maximum entre 2 classes étant inférieur à 3
- Pas de déséquilibre susceptible d'empêcher l'apprentissage d'un modèle de machine learning

## Etude ethique, disparate impact

In [69]:
# DI nationalite_hors_ue
df["outcome_favorable"] = df["classe_retour_emploi"].isin([0,1]).astype(int)

taux_reference = df[df["nationalite_hors_ue"] == 0]["outcome_favorable"].mean()
taux_protege = df[df["nationalite_hors_ue"] == 1]["outcome_favorable"].mean()

di = taux_protege / taux_reference

print(f"Disparate Impact nationalite_hors_ue : {di:.3f}")

Disparate Impact nationalite_hors_ue : 0.733


In [74]:
# Repartitiion de la variable age
df["tranche_age"] = pd.cut(df["age"], 
                           bins=[18, 30, 40, 50, 60, 70], 
                           labels=["18-29", "30-39", "40-49", "50-59", "60+"], 
                           right=False)


df.groupby("tranche_age")["outcome_favorable"].mean().mul(100).round(2)


tranche_age
18-29    85.85
30-39    91.55
40-49    91.32
50-59    66.67
60+      61.03
Name: outcome_favorable, dtype: float64

In [75]:
# Répartition de la variable cible favorable par tranche d'age
pd.crosstab(df["tranche_age"], df["classe_retour_emploi"], normalize="index").mul(100).round(2)

classe_retour_emploi,0,1,2
tranche_age,,,
18-29,41.64,44.21,14.15
30-39,52.58,38.97,8.45
40-49,49.62,41.70,8.68
50-59,15.93,50.73,33.33
60+,10.26,50.77,38.97


In [ ]:
# DI de la variable age
taux_ref = (df.loc[df["tranche_age"] == "30-39", "outcome_favorable"].mean())

di_age = (df.groupby("tranche_age")["outcome_favorable"].mean() / taux_ref).round(3)

display(di_age)

tranche_age
18-29    0.938
30-39    1.000
40-49    0.998
50-59    0.728
60+      0.667
Name: outcome_favorable, dtype: float64

## Conclusion
- Le tableau de répartition de retour à l'emploi par tranche d'age montre que les 30-49 ans ont le taux de retour à l'emploi inférieur à 12 mois le plus favorable, suivi par les moins de 29 ans. A partir de 50 ans, la proportion de retour à l'emploi après 12 mois dépasse 33% et même 38% pour les plus de 60 ans
- le calcul du disparate est de 0.728 pour la tranche d'age 50-59 et 0.667 pour le splus de 60 ans, ce qui constitue un signal d'alerte pour l'utilisation de cette donnée lors de l'entrainement

## DI croisé age / nationalité

In [ ]:
df["groupe_age_nat"] = (df["tranche_age"].astype(str) + "_" + df["nationalite_hors_ue"].astype(str))

resultats = (df.groupby("groupe_age_nat").agg(effectif=("outcome_favorable", "size"),
                                              taux_favorable=("outcome_favorable", "mean")))

ref = resultats["taux_favorable"].max()

resultats["DI"] = resultats["taux_favorable"] / ref

resultats.sort_values("DI")

,effectif,taux_favorable,DI
groupe_age_nat,,,
60+ | 1,25,0.080000,0.085854
50-59 | 1,58,0.241379,0.259041
60+ | 0,170,0.688235,0.738594
50-59 | 0,488,0.717213,0.769692
30-39 | 1,45,0.755556,0.810840
18-29 | 1,93,0.774194,0.830842
18-29 | 0,529,0.873346,0.937249
40-49 | 1,61,0.901639,0.967613
40-49 | 0,469,0.914712,0.981642


## Conclusion

L'analyse des Disparate Impacts individuels met en évidence des disparités modérées liées à l'âge et à la nationalité. Les catégories "50-59 ans", "60 ans et plus" ainsi que la population "hors UE" présentent des valeurs inférieures au seuil d'alerte de 0,8.

On constate un phénomène d'intersectionnalité. La combinaison "âge élevé" et "nationalité hors UE" entraîne une diminution beaucoup plus forte du taux de retour à l'emploi favorable que chacun de ces facteurs pris individuellement. C'est le cas pour les groupes "50-59 ans hors UE" (DI = 0,259 ; effectif = 58) et "60 ans et plus hors UE" (DI = 0,086 ; effectif = 25)

Ces résultats montrent un effet d'interaction entre l'âge et et la nationalité sur le jeu de données, ce qui constitue un risque de propagation ou d'amplification de ce biais pour un modèle entraîné sur ces données

# Conclusion Globale

- le dataset à un volume faible (2500 échantillons)
- la variable cible  est classe_retour_emploi
- la variable usager_id ne sera pas retenue car elle présente un risque de réidentification et n'apporte pas d'information métier
- la variable code_insee_commune comprends 2407 valeurs différentes sur 2500 échantillons. L'interprétation métier de cette variable est difficile et sa forte cardinalité rend son exploitation difficile. Elle ne sera donc pas retenue pour l'apprentissage
- Les variables age et nationalité_hors_ue présentent individuellement une DI légèrement inférieure au seuil de 0.8. L'analyse croisée de ces deux variables montre une disparité beaucoup plus important (DI = 0.08 pour le groupe le plus défavorisé, individus de plus de 60 ans ayant une nationalité hors UE). La nationalité sera donc exclue de l'apprentissage pour éviter le risque de propager ces disparités dans le modèle